新增医院数据 后续处理成医院-坐标数据的全过程：1、百度坐标和高德坐标系转换成WGS坐标，计算两套坐标系下坐标的距离。2、加入工商数据，得到较为齐全的医院坐标数据


In [1]:
# 百度坐标&高德坐标转换成WGS84坐标 eviltransform方法

import pandas as pd
import eviltransform
import logging
from pathlib import Path
from enum import Enum

In [ ]:
import pandas as pd
from pathlib import Path
import logging
from enum import Enum
import eviltransform

# ------------------------------
# 坐标系统枚举
# ------------------------------
class CoordSystem(Enum):
    AMAP = "amap"      # 高德 GCJ-02
    BAIDU = "baidu"    # 百度 BD-09

# ------------------------------
# 配置区
# ------------------------------
class Config:
    def __init__(self):
        self.base_dir = Path(
            r"D:\Hospital_Pharmacy program\data\hospital\hosp_registered_data\ol_data"
        )
        
        self.input_file = self.base_dir / "04_added_hospital_baidu_gaode_result2.csv"  #匹配了 省份 地区 名称的数据
        self.output_file = self.base_dir / "04_wgs84_added_hospital_baidu_gaode_result.csv"
        
        self.columns_map = {
            CoordSystem.BAIDU: {
                "lng": "百度经度",
                "lat": "百度纬度",
                "out_lng": "百度_wgs84经度",
                "out_lat": "百度_wgs84纬度",
            },
            CoordSystem.AMAP: {
                "lng": "高德经度",
                "lat": "高德纬度",
                "out_lng": "高德_wgs84经度",
                "out_lat": "高德_wgs84纬度",
            },
        }
        
        logging.basicConfig(
            level=logging.INFO,
            format="%(asctime)s - %(levelname)s - %(message)s"
        )

config = Config()

# ------------------------------
# 加载文件（增强版本）
# ------------------------------
def load_file(path: Path) -> pd.DataFrame:
    # 尝试更全面的编码列表
    encodings = [
        "utf-8-sig", "utf-8", "gbk", "gb2312", "gb18030", 
        "ansi", "iso-8859-1", "latin1", "cp1252", "cp1250",
        "big5", "shift_jis", "euc-jp", "euc-kr"
    ]
    
    # 首先检查文件是否存在
    if not path.exists():
        raise FileNotFoundError(f"文件不存在：{path}")
    
    print(f"尝试加载文件：{path}")
    print(f"文件大小：{path.stat().st_size} 字节")
    
    # 尝试读取前几行来查看内容
    try:
        with open(path, 'rb') as f:
            sample = f.read(500)
            print(f"文件前500字节（十六进制）：{sample[:100].hex()}...")
            print(f"文件前500字节（文本尝试解码为utf-8）：{sample[:100].decode('utf-8', errors='ignore')}")
    except Exception as e:
        print(f"无法读取文件样本：{e}")
    
    for enc in encodings:
        try:
            print(f"尝试使用编码：{enc}")
            df = pd.read_csv(path, encoding=enc, low_memory=False)
            print(f"使用编码 {enc} 成功读取文件")
            print(f"数据形状：{df.shape}")
            print(f"列名：{list(df.columns)}")
            return df
        except UnicodeDecodeError as e:
            print(f"编码 {enc} 失败：{str(e)[:100]}")
            continue
        except Exception as e:
            print(f"编码 {enc} 读取失败（非解码错误）：{str(e)[:100]}")
            continue
    
    # 如果所有编码都失败，尝试使用 errors='ignore' 或 'replace'
    print("尝试使用 errors='ignore' 方式读取...")
    for enc in ["utf-8", "gbk", "ansi"]:
        try:
            df = pd.read_csv(path, encoding=enc, encoding_errors='ignore', low_memory=False)
            print(f"使用编码 {enc} 和 errors='ignore' 成功读取文件")
            print(f"数据形状：{df.shape}")
            return df
        except Exception:
            continue
    
    # 最后尝试使用二进制模式读取
    print("尝试使用二进制模式读取...")
    try:
        df = pd.read_csv(path, encoding='latin1', low_memory=False)
        print("使用 latin1 编码成功读取文件")
        return df
    except Exception as e:
        raise ValueError(f"无法解析文件编码：{path}，已尝试所有编码方式。错误：{e}")

# ------------------------------
# 单行坐标转换
# ------------------------------
def convert_coord(row, coord_type: CoordSystem, col_map: dict):
    lng = row[col_map["lng"]]
    lat = row[col_map["lat"]]
    
    if pd.isna(lng) or pd.isna(lat):
        return None, None
    
    try:
        if coord_type == CoordSystem.AMAP:
            wgs_lat, wgs_lng = eviltransform.gcj2wgs(lat, lng)
        else:
            wgs_lat, wgs_lng = eviltransform.bd2wgs(lat, lng)
        
        return wgs_lng, wgs_lat
    except Exception as e:
        print(f"坐标转换失败: ({lat}, {lng}), 类型: {coord_type}, 错误: {e}")
        return None, None

# ------------------------------
# 主流程
# ------------------------------
def process():
    print("=" * 50)
    print("开始覆盖更新 WGS84 坐标")
    print("=" * 50)
    
    try:
        df = load_file(config.input_file)
        print("文件加载成功")
        
        for coord_type, col_map in config.columns_map.items():
            print(f"\n处理 {coord_type.value} 坐标系统...")
            
            required_cols = [
                col_map["lng"],
                col_map["lat"],
                col_map["out_lng"],
                col_map["out_lat"],
            ]
            
            # 检查列是否存在
            missing_cols = [col for col in required_cols if col not in df.columns]
            if missing_cols:
                print(f"警告：文件中缺少列：{missing_cols}")
                print(f"现有列：{list(df.columns)}")
                
                # 尝试寻找相似的列名
                for missing_col in missing_cols:
                    similar_cols = [col for col in df.columns if missing_col[:4] in col or missing_col[-4:] in col]
                    if similar_cols:
                        print(f"  可能相似的列：{similar_cols}")
                
                # 如果缺少输入列，跳过这个坐标系统
                if col_map["lng"] not in df.columns or col_map["lat"] not in df.columns:
                    print(f"跳过 {coord_type.value}，因为缺少输入列")
                    continue
            
            # 确保输出列存在
            for out_col in [col_map["out_lng"], col_map["out_lat"]]:
                if out_col not in df.columns:
                    print(f"添加缺失的输出列：{out_col}")
                    df[out_col] = None
            
            # 应用转换
            df[[col_map["out_lng"], col_map["out_lat"]]] = df.apply(
                lambda row: pd.Series(
                    convert_coord(row, coord_type, col_map)
                ),
                axis=1
            )
            
            print(f"已更新 {coord_type.value} → WGS84")
            print(f"成功转换数量：{df[col_map['out_lng']].notna().sum()}")
        
        # 保存文件
        df.to_csv(config.output_file, index=False, encoding="utf-8-sig")
        print(f"\n文件已保存：{config.output_file}")
        print(f"保存成功，总行数：{len(df)}")
        
    except Exception as e:
        print(f"处理过程中发生错误：{e}")
        import traceback
        traceback.print_exc()

# ------------------------------
if __name__ == "__main__":
    process()

开始覆盖更新 WGS84 坐标
尝试加载文件：D:\Hospital_Pharmacy program\data\hospital\hosp_registered_data\ol_data\04_added_hospital_baidu_gaode_result2.csv
文件大小：889074 字节
文件前500字节（十六进制）：636f6d70616e7969642c5052495049442c454e544e414d452c554e49534349442c5245474e4f2c4e4143414f49442c4e414d452c4e414d455449544c452c454e54545950452c4553444154452c41505052444154452c454e545354415455532c52454743...
文件前500字节（文本尝试解码为utf-8）：companyid,PRIPID,ENTNAME,UNISCID,REGNO,NACAOID,NAME,NAMETITLE,ENTTYPE,ESDATE,APPRDATE,ENTSTATUS,REGC
尝试使用编码：utf-8-sig
编码 utf-8-sig 失败：'utf-8' codec can't decode byte 0xb0 in position 222: invalid start byte
尝试使用编码：utf-8
编码 utf-8 失败：'utf-8' codec can't decode byte 0xb0 in position 0: invalid start byte
尝试使用编码：gbk
使用编码 gbk 成功读取文件
数据形状：(1994, 36)
列名：['companyid', 'PRIPID', 'ENTNAME', 'UNISCID', 'REGNO', 'NACAOID', 'NAME', 'NAMETITLE', 'ENTTYPE', 'ESDATE', 'APPRDATE', 'ENTSTATUS', 'REGCAP', 'REGCAP_NAME', 'REGCAPCUR', 'RECCAP', 'REGORG', 'OPFROM', 'OPTO', 'OPSCOPE', 'DOM', 'DOMDISTRICT', 'NIC_ID', 'CAND

In [4]:

#1. 对两套经纬度数据差异在500m以下的，认为其定位均准确，此处选取百度API获得的经纬度数据
#2. 对于两套经纬度数据差异在500m以上的，认为定位有误，利用openstreetmap进行逆向编码爬取，获得各自经纬度下的对应地址
#3. 将对应地址名称与医院名称进行模糊匹配，模糊匹配分数在80分以上的认为定位正确，选取定位正确的API对应获得的经纬度数据

import pandas as pd
import math
import numpy as np
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm
import requests
from fuzzywuzzy import process
import re
import os
import pickle


d:\swx\Anaconda\Lib\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [5]:

import pandas as pd
import numpy as np
import os

# ------------------------------
# 文件读取函数
# ------------------------------
def robust_read_csv(file_path):
    """
    尝试多种编码读取CSV文件
    """
    # 常见的编码列表（按优先级排序）
    encodings = [
        'utf-8-sig',  # 带BOM的UTF-8
        'utf-8',      # 标准UTF-8
        'gbk',        # 简体中文Windows
        'gb2312',     # 简体中文
        'gb18030',    # 中文国家标准
        'latin1',     # ISO-8859-1
        'iso-8859-1', # 另一种拉丁编码
        'cp1252',     # Windows拉丁编码
    ]
    
    for encoding in encodings:
        try:
            print(f"尝试使用 {encoding} 编码读取...")
            df = pd.read_csv(file_path, encoding=encoding)
            print(f"✓ 使用 {encoding} 编码读取成功，共 {len(df)} 行，{len(df.columns)} 列")
            return df
        except UnicodeDecodeError:
            continue
        except Exception as e:
            print(f"使用 {encoding} 编码时出错: {str(e)}")
            continue
    
    # 如果所有编码都失败，使用带错误处理的读取
    print("所有编码尝试失败，使用错误忽略模式读取...")
    return pd.read_csv(file_path, encoding='utf-8', errors='ignore')

# ------------------------------
# 数据加载
# ------------------------------
file_path = r"D:\Hospital_Pharmacy program\data\hospital\hosp_registered_data\ol_data\04_wgs84_added_hospital_baidu_gaode_result.csv"

if os.path.exists(file_path):
    print(f"文件存在，大小: {os.path.getsize(file_path)} 字节")
    df_merged = robust_read_csv(file_path)
else:
    print("错误：文件不存在！")
    exit()
print(f"数据加载完成，共 {len(df_merged)} 条记录")
print("列名:", df_merged.columns.tolist())
print("\n前几行数据:")
print(df_merged.head())

# ------------------------------
# Haversine 距离函数
# ------------------------------
def haversine_distance(lon1, lat1, lon2, lat2):
    """
    计算两个 WGS84 坐标点之间的地面距离（米）
    """
    # 任一坐标缺失 → 返回 NaN
    if pd.isna(lon1) or pd.isna(lat1) or pd.isna(lon2) or pd.isna(lat2):
        return np.nan

    # 转弧度
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )
    c = 2 * np.arcsin(np.sqrt(a))
    r = 6371000  # 地球平均半径（米）

    return c * r


# ------------------------------
# 距离计算（百度 vs 高德）
# ------------------------------
df_merged["distance"] = df_merged.apply(
    lambda row: haversine_distance(
        row["百度_wgs84经度"], row["百度_wgs84纬度"],
        row["高德_wgs84经度"], row["高德_wgs84纬度"],
    ),
    axis=1
)

# ------------------------------
# 距离统计
# ------------------------------
distance_stats = df_merged["distance"].describe()
print("百度 vs 高德 WGS84 距离偏差统计：")
print(distance_stats)


# ------------------------------
# 基础字段类型规范（防止后续 groupby / 导出问题）
# ------------------------------
for col in ["ENTNAME", "省份"]:
    if col in df_merged.columns:
        df_merged[col] = df_merged[col].astype(str)


# ------------------------------
# 大偏差筛选
# ------------------------------
THRESHOLD = 3_000_000  # 3000 km，理论上异常值
large_deviation = df_merged[df_merged["distance"] > THRESHOLD]

print(f"偏差超过 {THRESHOLD} 米的记录有 {len(large_deviation)} 条")


文件存在，大小: 1254829 字节
尝试使用 utf-8-sig 编码读取...
✓ 使用 utf-8-sig 编码读取成功，共 1994 行，40 列
数据加载完成，共 1994 条记录
列名: ['companyid', 'PRIPID', 'ENTNAME', 'UNISCID', 'REGNO', 'NACAOID', 'NAME', 'NAMETITLE', 'ENTTYPE', 'ESDATE', 'APPRDATE', 'ENTSTATUS', 'REGCAP', 'REGCAP_NAME', 'REGCAPCUR', 'RECCAP', 'REGORG', 'OPFROM', 'OPTO', 'OPSCOPE', 'DOM', 'DOMDISTRICT', 'NIC_ID', 'CANDATE', 'REVDATE', 'created', 'updated', 'lat', 'lng', '百度经度', '百度纬度', '高德经度', '高德纬度', 'admin_code', '省份', '地区', '百度_wgs84经度', '百度_wgs84纬度', '高德_wgs84经度', '高德_wgs84纬度']

前几行数据:
   companyid                                             PRIPID  \
0    1176224                                                NaN   
1    2834800      PROVINCENODENUM370000370902000022018122417963   
2    4653840      PROVINCENODENUM210000210103001012021061839646   
3   10846160                                                NaN   
4   13401408  PROVINCENODENUM110000ff8080815b27acb4015b3cb11...   

                   ENTNAME             UNISCID            REGNO 

In [6]:
from tqdm import tqdm
tqdm.pandas()

from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from geopy.exc import GeocoderTimedOut
import time
import requests
import pandas as pd

# ------------------------------
# 逆地理编码（带重试）
# ------------------------------
def get_address_details_with_retry(lat, lng, retries=3, delay=2):
    for attempt in range(retries):
        try:
            location = geocode(f"{lat}, {lng}", exactly_one=True, language='zh')
            return location.raw.get('address', {}) if location else {}
        except GeocoderTimedOut:
            print(f"第 {attempt + 1} 次尝试超时，{delay} 秒后重试...")
            time.sleep(delay)
        except Exception as e:
            print(f"第 {attempt + 1} 次尝试出现异常: {str(e)}")
            break
    return {}


# ------------------------------
# 初始化地理编码器
# ------------------------------
geolocator = Nominatim(
    user_agent="hospital_geo_validation",
    timeout=30
)

geocode = RateLimiter(geolocator.reverse, min_delay_seconds=1)


def get_address_details(lat, lng):
    try:
        location = geocode(f"{lat}, {lng}", exactly_one=True, language='zh')
        return location.raw.get('address', {}) if location else {}
    except Exception as e:
        print(f"逆地理编码失败: {str(e)}")
        return {}


# ------------------------------
# 测试 Nominatim 服务
# ------------------------------
try:
    test_url = "https://nominatim.openstreetmap.org/reverse?lat=39.9&lon=116.4&format=json"
    response = requests.get(test_url, timeout=5)
    print("Nominatim 服务正常" if response.status_code == 200 else "服务返回错误")
except Exception as e:
    print(f"无法连接 Nominatim: {str(e)}")

try:
    location = geolocator.reverse("31.00779569556785,121.40279835486228", language='zh')
    print("测试查询成功:", location.address)
except Exception as e:
    print(f"测试查询失败: {str(e)}")


# ------------------------------
# 省份边界数据（示例）
# ------------------------------
province_bounds = {
    "北京市": {"min_lon": 115.42, "max_lon": 117.51, "min_lat": 39.44, "max_lat": 41.06},
    "天津市": {"min_lon": 116.71, "max_lon": 118.07, "min_lat": 38.56, "max_lat": 40.25},
    # 其他省份边界保持原样
}


def check_province(lon, lat, target_province):
    bounds = province_bounds.get(target_province)
    if not bounds or pd.isna(lon) or pd.isna(lat):
        return False
    return (
        bounds['min_lon'] <= lon <= bounds['max_lon']
        and bounds['min_lat'] <= lat <= bounds['max_lat']
    )


# ------------------------------
# 省份一致性检查
# ------------------------------
df_merged['province_check_gd'] = df_merged.apply(
    lambda row: check_province(
        row['高德_wgs84经度'],
        row['高德_wgs84纬度'],
        row['省份']
    ),
    axis=1
)

df_merged['province_check_bd'] = df_merged.apply(
    lambda row: check_province(
        row['百度_wgs84经度'],
        row['百度_wgs84纬度'],
        row['省份']
    ),
    axis=1
)

print("✅ 省份边界检查完成")


服务返回错误
测试查询成功: 上海市第五人民医院, 鹤庆路, 闵行, 江川路街道, 闵行区, 上海市, 200240, 中国
✅ 省份边界检查完成


In [9]:
import os
import pickle

# 第六部分：偏差数据分析和逆地理编码
THRESHOLD_MIN = 500
THRESHOLD_MAX = 5000

# 筛选不同阈值范围的偏差数据
large_deviation_500_5000 = df_merged[
    (df_merged['distance'] > THRESHOLD_MIN) &
    (df_merged['distance'] <= THRESHOLD_MAX)
].copy()

large_deviation_5000 = df_merged[
    df_merged['distance'] > THRESHOLD_MAX
].copy()

print(f"中等偏差记录(500–5000米): {len(large_deviation_500_5000)}")
print(f"大偏差记录(>5000米): {len(large_deviation_5000)}")


# 逆地理编码缓存路径
BASE_DIR = r"D:\Hospital_Pharmacy program\data\hospital\hosp_registered_data\ol_data"
GEOCODE_CACHE_FILE = os.path.join(BASE_DIR, "geocode_cache.pkl")

# 如果文件夹不存在，则自动创建
os.makedirs(BASE_DIR, exist_ok=True)


def load_geocode_cache():
    if os.path.exists(GEOCODE_CACHE_FILE):
        try:
            with open(GEOCODE_CACHE_FILE, "rb") as f:
                cache = pickle.load(f)
                print(f"已加载逆地理编码缓存，包含 {len(cache)} 条记录")
                return cache
        except Exception as e:
            print(f"加载缓存失败: {e}")
    return {}


def save_geocode_cache(cache):
    try:
        with open(GEOCODE_CACHE_FILE, "wb") as f:
            pickle.dump(cache, f)
        print(f"逆地理编码缓存已保存，包含 {len(cache)} 条记录")
    except Exception as e:
        print(f"保存缓存失败: {e}")


def get_address_details_with_cache(lat, lng, cache):
    if pd.isna(lat) or pd.isna(lng):
        return None

    cache_key = f"{lat:.6f},{lng:.6f}"

    if cache_key in cache:
        return cache[cache_key]

    address_details = get_address_details(lat, lng)
    cache[cache_key] = address_details

    if len(cache) % 10 == 0:
        save_geocode_cache(cache)

    return address_details


# 加载缓存
geocode_cache = load_geocode_cache()


# 中等偏差记录的逆地理编码
if len(large_deviation_500_5000) > 0:
    print("开始处理中等偏差记录的逆地理编码...")

    large_deviation_500_5000["gd_address"] = large_deviation_500_5000.apply(
        lambda row: get_address_details_with_cache(
            row["高德_wgs84纬度"],
            row["高德_wgs84经度"],
            geocode_cache
        ),
        axis=1
    )

    large_deviation_500_5000["bd_address"] = large_deviation_500_5000.apply(
        lambda row: get_address_details_with_cache(
            row["百度_wgs84纬度"],
            row["百度_wgs84经度"],
            geocode_cache
        ),
        axis=1
    )


# 大偏差记录的逆地理编码
if len(large_deviation_5000) > 0:
    print("开始处理大偏差记录的逆地理编码...")

    large_deviation_5000["gd_address"] = large_deviation_5000.apply(
        lambda row: get_address_details_with_cache(
            row["高德_wgs84纬度"],
            row["高德_wgs84经度"],
            geocode_cache
        ),
        axis=1
    )

    large_deviation_5000["bd_address"] = large_deviation_5000.apply(
        lambda row: get_address_details_with_cache(
            row["百度_wgs84纬度"],
            row["百度_wgs84经度"],
            geocode_cache
        ),
        axis=1
    )


# 保存缓存
save_geocode_cache(geocode_cache)


# 导出结果
large_deviation_500_5000.to_csv(
    os.path.join(BASE_DIR, "large_deviation_500_5000.csv"),
    index=False,
    encoding="utf-8-sig"
)

large_deviation_5000.to_csv(
    os.path.join(BASE_DIR, "large_deviation_5000.csv"),
    index=False,
    encoding="utf-8-sig"
)

print("逆地理编码完成")

中等偏差记录(500–5000米): 334
大偏差记录(>5000米): 800
开始处理中等偏差记录的逆地理编码...
逆地理编码缓存已保存，包含 10 条记录
逆地理编码缓存已保存，包含 20 条记录
逆地理编码缓存已保存，包含 30 条记录
逆地理编码缓存已保存，包含 40 条记录
逆地理编码缓存已保存，包含 50 条记录
逆地理编码缓存已保存，包含 60 条记录
逆地理编码缓存已保存，包含 70 条记录
逆地理编码缓存已保存，包含 80 条记录
逆地理编码缓存已保存，包含 90 条记录
逆地理编码缓存已保存，包含 100 条记录
逆地理编码缓存已保存，包含 110 条记录
逆地理编码缓存已保存，包含 120 条记录
逆地理编码缓存已保存，包含 130 条记录
逆地理编码缓存已保存，包含 140 条记录
逆地理编码缓存已保存，包含 150 条记录
逆地理编码缓存已保存，包含 160 条记录
逆地理编码缓存已保存，包含 170 条记录
逆地理编码缓存已保存，包含 180 条记录
逆地理编码缓存已保存，包含 190 条记录
逆地理编码缓存已保存，包含 200 条记录
逆地理编码缓存已保存，包含 210 条记录
逆地理编码缓存已保存，包含 220 条记录
逆地理编码缓存已保存，包含 230 条记录
逆地理编码缓存已保存，包含 240 条记录
逆地理编码缓存已保存，包含 250 条记录
逆地理编码缓存已保存，包含 260 条记录
逆地理编码缓存已保存，包含 270 条记录
逆地理编码缓存已保存，包含 280 条记录
逆地理编码缓存已保存，包含 290 条记录
逆地理编码缓存已保存，包含 300 条记录
逆地理编码缓存已保存，包含 310 条记录
逆地理编码缓存已保存，包含 320 条记录
逆地理编码缓存已保存，包含 330 条记录
逆地理编码缓存已保存，包含 340 条记录
逆地理编码缓存已保存，包含 350 条记录
逆地理编码缓存已保存，包含 360 条记录
逆地理编码缓存已保存，包含 370 条记录
逆地理编码缓存已保存，包含 380 条记录
逆地理编码缓存已保存，包含 390 条记录
逆地理编码缓存已保存，包含 400 条记录
逆地理编码缓存已保存，包含 410 条记录
逆地理编码缓存已保存，包含 420 条记录
逆地理编码缓存已保存，包含 430 条记录
逆

2026-07-02 18:51:08,192 - WARNING - Retrying (Retry(total=1, connect=None, read=None, redirect=None, status=None)) after connection broken by 'ReadTimeoutError("HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Read timed out. (read timeout=30)")': /reverse?lat=23.15267103884281&lon=113.51379438994924&format=json&accept-language=zh&addressdetails=1
2026-07-02 18:51:20,023 - WARNING - Retrying (Retry(total=0, connect=None, read=None, redirect=None, status=None)) after connection broken by 'SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1028)')': /reverse?lat=23.15267103884281&lon=113.51379438994924&format=json&accept-language=zh&addressdetails=1


逆地理编码缓存已保存，包含 740 条记录
逆地理编码缓存已保存，包含 750 条记录
逆地理编码缓存已保存，包含 760 条记录
逆地理编码缓存已保存，包含 770 条记录
逆地理编码缓存已保存，包含 780 条记录
逆地理编码缓存已保存，包含 790 条记录
逆地理编码缓存已保存，包含 800 条记录
逆地理编码缓存已保存，包含 810 条记录
逆地理编码缓存已保存，包含 820 条记录
逆地理编码缓存已保存，包含 830 条记录
逆地理编码缓存已保存，包含 840 条记录
逆地理编码缓存已保存，包含 850 条记录
逆地理编码缓存已保存，包含 860 条记录
逆地理编码缓存已保存，包含 870 条记录
逆地理编码缓存已保存，包含 880 条记录
逆地理编码缓存已保存，包含 890 条记录
逆地理编码缓存已保存，包含 900 条记录
逆地理编码缓存已保存，包含 910 条记录
逆地理编码缓存已保存，包含 920 条记录
逆地理编码缓存已保存，包含 930 条记录
逆地理编码缓存已保存，包含 940 条记录
逆地理编码缓存已保存，包含 950 条记录
逆地理编码缓存已保存，包含 960 条记录
逆地理编码缓存已保存，包含 970 条记录
逆地理编码缓存已保存，包含 980 条记录
逆地理编码缓存已保存，包含 990 条记录
逆地理编码缓存已保存，包含 1000 条记录
逆地理编码缓存已保存，包含 1010 条记录
逆地理编码缓存已保存，包含 1020 条记录
逆地理编码缓存已保存，包含 1030 条记录
逆地理编码缓存已保存，包含 1040 条记录
逆地理编码缓存已保存，包含 1050 条记录
逆地理编码缓存已保存，包含 1060 条记录
逆地理编码缓存已保存，包含 1070 条记录
逆地理编码缓存已保存，包含 1080 条记录
逆地理编码缓存已保存，包含 1090 条记录


2026-07-02 18:58:05,975 - WARNING - Retrying (Retry(total=1, connect=None, read=None, redirect=None, status=None)) after connection broken by 'ReadTimeoutError("HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Read timed out. (read timeout=30)")': /reverse?lat=38.64092763112551&lon=113.14710079958974&format=json&accept-language=zh&addressdetails=1
2026-07-02 18:58:36,002 - WARNING - Retrying (Retry(total=0, connect=None, read=None, redirect=None, status=None)) after connection broken by 'ReadTimeoutError("HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Read timed out. (read timeout=30)")': /reverse?lat=38.64092763112551&lon=113.14710079958974&format=json&accept-language=zh&addressdetails=1
2026-07-02 18:59:06,027 - WARNING - RateLimiter caught an error, retrying (0/2 tries). Called with (*('38.64092763112551, 113.14710079958974',), **{'exactly_one': True, 'language': 'zh'}).
Traceback (most recent call last):
  File "d:\swx\Anaconda\Lib\site-package

逆地理编码缓存已保存，包含 1100 条记录
逆地理编码缓存已保存，包含 1110 条记录
逆地理编码缓存已保存，包含 1120 条记录
逆地理编码缓存已保存，包含 1130 条记录
逆地理编码缓存已保存，包含 1140 条记录
逆地理编码缓存已保存，包含 1150 条记录
逆地理编码缓存已保存，包含 1160 条记录
逆地理编码缓存已保存，包含 1170 条记录
逆地理编码缓存已保存，包含 1180 条记录
逆地理编码缓存已保存，包含 1190 条记录
逆地理编码缓存已保存，包含 1200 条记录
逆地理编码缓存已保存，包含 1210 条记录
逆地理编码缓存已保存，包含 1220 条记录
逆地理编码缓存已保存，包含 1230 条记录
逆地理编码缓存已保存，包含 1240 条记录
逆地理编码缓存已保存，包含 1250 条记录
逆地理编码缓存已保存，包含 1260 条记录
逆地理编码缓存已保存，包含 1270 条记录
逆地理编码缓存已保存，包含 1280 条记录


2026-07-02 19:03:31,658 - WARNING - Retrying (Retry(total=1, connect=None, read=None, redirect=None, status=None)) after connection broken by 'ReadTimeoutError("HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Read timed out. (read timeout=30)")': /reverse?lat=40.631195833095866&lon=117.36445781796678&format=json&accept-language=zh&addressdetails=1
2026-07-02 19:03:41,794 - WARNING - Retrying (Retry(total=0, connect=None, read=None, redirect=None, status=None)) after connection broken by 'SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1028)')': /reverse?lat=40.631195833095866&lon=117.36445781796678&format=json&accept-language=zh&addressdetails=1


逆地理编码缓存已保存，包含 1290 条记录
逆地理编码缓存已保存，包含 1300 条记录
逆地理编码缓存已保存，包含 1310 条记录
逆地理编码缓存已保存，包含 1320 条记录
逆地理编码缓存已保存，包含 1330 条记录
逆地理编码缓存已保存，包含 1340 条记录
逆地理编码缓存已保存，包含 1350 条记录
逆地理编码缓存已保存，包含 1360 条记录
逆地理编码缓存已保存，包含 1370 条记录


2026-07-02 19:05:22,061 - WARNING - Retrying (Retry(total=1, connect=None, read=None, redirect=None, status=None)) after connection broken by 'ConnectionResetError(10054, '远程主机强迫关闭了一个现有的连接。', None, 10054, None)': /reverse?lat=30.66564226966962&lon=113.15997996648711&format=json&accept-language=zh&addressdetails=1


逆地理编码缓存已保存，包含 1380 条记录
逆地理编码缓存已保存，包含 1390 条记录
逆地理编码缓存已保存，包含 1400 条记录
逆地理编码缓存已保存，包含 1410 条记录
逆地理编码缓存已保存，包含 1420 条记录
逆地理编码缓存已保存，包含 1430 条记录
逆地理编码缓存已保存，包含 1440 条记录
逆地理编码缓存已保存，包含 1450 条记录
逆地理编码缓存已保存，包含 1460 条记录
逆地理编码缓存已保存，包含 1470 条记录
逆地理编码缓存已保存，包含 1480 条记录
逆地理编码缓存已保存，包含 1490 条记录
逆地理编码缓存已保存，包含 1500 条记录
逆地理编码缓存已保存，包含 1510 条记录
逆地理编码缓存已保存，包含 1520 条记录
逆地理编码缓存已保存，包含 1530 条记录
逆地理编码缓存已保存，包含 1540 条记录
逆地理编码缓存已保存，包含 1550 条记录
逆地理编码缓存已保存，包含 1560 条记录
逆地理编码缓存已保存，包含 1570 条记录
逆地理编码缓存已保存，包含 1580 条记录
逆地理编码缓存已保存，包含 1590 条记录
逆地理编码缓存已保存，包含 1600 条记录
逆地理编码缓存已保存，包含 1610 条记录
逆地理编码缓存已保存，包含 1620 条记录
逆地理编码缓存已保存，包含 1630 条记录
逆地理编码缓存已保存，包含 1640 条记录
逆地理编码缓存已保存，包含 1650 条记录
逆地理编码缓存已保存，包含 1660 条记录
逆地理编码缓存已保存，包含 1670 条记录
逆地理编码缓存已保存，包含 1680 条记录
逆地理编码缓存已保存，包含 1690 条记录
逆地理编码缓存已保存，包含 1700 条记录
逆地理编码缓存已保存，包含 1710 条记录
逆地理编码缓存已保存，包含 1720 条记录
逆地理编码缓存已保存，包含 1730 条记录
逆地理编码缓存已保存，包含 1740 条记录
逆地理编码缓存已保存，包含 1750 条记录
逆地理编码缓存已保存，包含 1760 条记录
逆地理编码缓存已保存，包含 1770 条记录
逆地理编码缓存已保存，包含 1780 条记录
逆地理编码缓存已保存，包含 1790 条记录
逆地理编码缓存已保存，包含 1800 条记录
逆地理编码缓存已保存，

## 2. 坐标偏差判断和逆地理编码模糊匹配

前一个 cell 已经对 >500m 的中偏差和大偏差记录完成逆地理编码。本 cell 不再调用逆地理编码，只使用已有的 gd_address / bd_address 做模糊匹配。

坐标选择顺序：

1. 百度和高德 WGS 坐标偏差 ≤500m 时，默认选用百度 WGS 坐标。
2. 偏差 >500m 时，用已有逆地理编码地址与医院名称模糊匹配，匹配成功则用对应 API 的 WGS 坐标。
3. 未匹配的记录后续用工商坐标补充，工商坐标视为高德/GCJ-02 坐标，需转为 WGS。
4. 工商坐标仍无法补充的记录，最后用百度 WGS 兜底，若百度 WGS 空缺则用高德 WGS。


In [12]:
import ast
import os
import re

import numpy as np
import pandas as pd
from rapidfuzz import fuzz

BASE_DIR = r"D:\Hospital_Pharmacy program\data\hospital\hosp_registered_data\ol_data"
INPUT_FILE = os.path.join(BASE_DIR, "04_wgs84_added_hospital_baidu_gaode_result.csv")
REVERSE_MEDIUM_FILE = os.path.join(BASE_DIR, "large_deviation_500_5000.csv")
REVERSE_LARGE_FILE = os.path.join(BASE_DIR, "large_deviation_5000.csv")
PRE_FALLBACK_FILE = os.path.join(BASE_DIR, "05_added_hospital_after_reverse_match.csv")
REVERSE_MATCH_FILE = os.path.join(BASE_DIR, "05_added_hospital_reverse_geocode_match_detail.csv")
UNMATCHED_REVERSE_FILE = os.path.join(BASE_DIR, "05_added_hospital_reverse_unmatched_for_business_fill.csv")

DISTANCE_CONSISTENT_M = 500
FUZZY_SCORE_THRESHOLD = 80


def read_csv_auto(path):
    for enc in ["utf-8-sig", "utf-8", "gb18030", "gbk"]:
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(path, encoding="utf-8", errors="ignore", low_memory=False)


if "df_merged" not in globals():
    df_merged = read_csv_auto(INPUT_FILE)


def pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"Missing required column. Tried: {candidates}")


BD_LON = pick_col(df_merged, ["百度_wgs84经度", "百度_wgs84经度1"])
BD_LAT = pick_col(df_merged, ["百度_wgs84纬度", "百度_wgs84纬度1"])
GD_LON = pick_col(df_merged, ["高德_wgs84经度", "高德_wgs84经度1"])
GD_LAT = pick_col(df_merged, ["高德_wgs84纬度", "高德_wgs84纬度1"])

if "place_merge_id" not in df_merged.columns:
    df_merged["place_merge_id"] = df_merged.index.astype(str)

for c in [BD_LON, BD_LAT, GD_LON, GD_LAT, "distance"]:
    if c in df_merged.columns:
        df_merged[c] = pd.to_numeric(df_merged[c], errors="coerce")

if "distance" not in df_merged.columns or df_merged["distance"].isna().all():
    def haversine_distance(lon1, lat1, lon2, lat2):
        if pd.isna(lon1) or pd.isna(lat1) or pd.isna(lon2) or pd.isna(lat2):
            return np.nan
        lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
        dlon = lon2 - lon1
        dlat = lat2 - lat1
        a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
        return 2 * 6371000 * np.arcsin(np.sqrt(a))

    df_merged["distance"] = df_merged.apply(
        lambda row: haversine_distance(row[BD_LON], row[BD_LAT], row[GD_LON], row[GD_LAT]),
        axis=1,
    )


def parse_address_value(value):
    if isinstance(value, dict):
        return value
    if pd.isna(value):
        return {}
    text = str(value).strip()
    if not text:
        return {}
    try:
        parsed = ast.literal_eval(text)
        return parsed if isinstance(parsed, dict) else {}
    except Exception:
        return {"display_name": text}


def flatten_address(addr):
    addr = parse_address_value(addr)
    values = []
    for key in [
        "amenity", "building", "office", "shop", "tourism", "healthcare",
        "road", "neighbourhood", "suburb", "village", "town", "city",
        "county", "state", "province", "display_name",
    ]:
        value = addr.get(key)
        if value and str(value) not in values:
            values.append(str(value))
    if not values:
        values = [str(v) for v in addr.values() if pd.notna(v)]
    return " ".join(values)


def clean_text(value):
    text = "" if pd.isna(value) else str(value)
    text = re.sub(r"[\s\W_]+", "", text, flags=re.UNICODE)
    text = text.replace("有限公司", "").replace("有限责任公司", "")
    text = text.replace("股份有限公司", "").replace("分公司", "")
    return text


def fuzzy_match_score(address_text, hospital_name):
    address_text = clean_text(address_text)
    hospital_name = clean_text(hospital_name)
    if not address_text or not hospital_name:
        return 0
    return max(
        fuzz.WRatio(hospital_name, address_text),
        fuzz.partial_ratio(hospital_name, address_text),
        fuzz.token_set_ratio(hospital_name, address_text),
    )


def load_reverse_geocoded_deviations():
    frames = []
    if "large_deviation_500_5000" in globals():
        frames.append(large_deviation_500_5000.copy())
    elif os.path.exists(REVERSE_MEDIUM_FILE):
        frames.append(read_csv_auto(REVERSE_MEDIUM_FILE))

    if "large_deviation_5000" in globals():
        frames.append(large_deviation_5000.copy())
    elif os.path.exists(REVERSE_LARGE_FILE):
        frames.append(read_csv_auto(REVERSE_LARGE_FILE))

    if not frames:
        raise FileNotFoundError("No reverse-geocoded deviation data found. Run the previous reverse-geocoding cell first.")

    out = pd.concat(frames, ignore_index=True)
    missing = {"gd_address", "bd_address"} - set(out.columns)
    if missing:
        raise KeyError(f"Reverse-geocoded data missing columns: {missing}. Run the previous cell first.")
    return out


def attach_place_merge_id(reverse_df):
    if "place_merge_id" in reverse_df.columns:
        return reverse_df

    reverse_df = reverse_df.copy()
    keys = [c for c in ["companyid", "PRIPID", "UNISCID", "REGNO"] if c in reverse_df.columns and c in df_merged.columns]

    for key in keys:
        mapper = (
            df_merged.dropna(subset=[key])
            .drop_duplicates(subset=[key], keep="first")
            .set_index(key)["place_merge_id"]
        )
        reverse_df["place_merge_id"] = reverse_df[key].map(mapper)
        if reverse_df["place_merge_id"].notna().sum() > 0:
            return reverse_df

    if "ENTNAME" in reverse_df.columns and "ENTNAME" in df_merged.columns:
        mapper = (
            df_merged.dropna(subset=["ENTNAME"])
            .drop_duplicates(subset=["ENTNAME"], keep="first")
            .set_index("ENTNAME")["place_merge_id"]
        )
        reverse_df["place_merge_id"] = reverse_df["ENTNAME"].map(mapper)

    return reverse_df


# 初始化输出列，显式控制 dtype，避免 pandas 3 严格 dtype 赋值报错。
for c in ["wgs_lon_true", "wgs_lat_true"]:
    if c not in df_merged.columns:
        df_merged[c] = np.nan
    df_merged[c] = pd.to_numeric(df_merged[c], errors="coerce").astype("float64")

for c in ["coord_source", "coord_rule"]:
    if c not in df_merged.columns:
        df_merged[c] = ""
    df_merged[c] = df_merged[c].astype("object")

if "needs_fallback" not in df_merged.columns:
    df_merged["needs_fallback"] = 0
df_merged["needs_fallback"] = (
    pd.to_numeric(df_merged["needs_fallback"], errors="coerce")
    .fillna(0)
    .astype("int64")
)

# 规则1：百度和高德 WGS 偏差 <=500m，直接采用百度 WGS。
consistent_mask = (
    df_merged["distance"].le(DISTANCE_CONSISTENT_M)
    & df_merged[BD_LON].notna()
    & df_merged[BD_LAT].notna()
)

df_merged.loc[consistent_mask, "wgs_lon_true"] = df_merged.loc[consistent_mask, BD_LON]
df_merged.loc[consistent_mask, "wgs_lat_true"] = df_merged.loc[consistent_mask, BD_LAT]
df_merged.loc[consistent_mask, "coord_source"] = "baidu"
df_merged.loc[consistent_mask, "coord_rule"] = "baidu_gaode_distance_le_500m"
df_merged.loc[consistent_mask, "needs_fallback"] = 0

# 规则2：>500m 的记录，使用前一 cell 已经逆地理编码好的地址做模糊匹配。
reverse_detail = attach_place_merge_id(load_reverse_geocoded_deviations())
reverse_detail = reverse_detail[reverse_detail["place_merge_id"].notna()].copy()

reverse_detail["gd_address_text"] = reverse_detail["gd_address"].apply(flatten_address)
reverse_detail["bd_address_text"] = reverse_detail["bd_address"].apply(flatten_address)

reverse_detail["gd_match_score"] = [
    fuzzy_match_score(addr, name)
    for addr, name in zip(reverse_detail["gd_address_text"], reverse_detail["ENTNAME"])
]

reverse_detail["bd_match_score"] = [
    fuzzy_match_score(addr, name)
    for addr, name in zip(reverse_detail["bd_address_text"], reverse_detail["ENTNAME"])
]


def choose_reverse_geocode_coord(row):
    gd_ok = row["gd_match_score"] >= FUZZY_SCORE_THRESHOLD
    bd_ok = row["bd_match_score"] >= FUZZY_SCORE_THRESHOLD

    if bd_ok and (row["bd_match_score"] >= row["gd_match_score"]):
        return pd.Series({
            "wgs_lon_true": row[BD_LON],
            "wgs_lat_true": row[BD_LAT],
            "coord_source": "baidu_reverse_geocode_match",
            "coord_rule": f"distance_gt_500m_bd_reverse_score_ge_{FUZZY_SCORE_THRESHOLD}",
            "needs_fallback": 0,
        })

    if gd_ok:
        return pd.Series({
            "wgs_lon_true": row[GD_LON],
            "wgs_lat_true": row[GD_LAT],
            "coord_source": "gaode_reverse_geocode_match",
            "coord_rule": f"distance_gt_500m_gd_reverse_score_ge_{FUZZY_SCORE_THRESHOLD}",
            "needs_fallback": 0,
        })

    return pd.Series({
        "wgs_lon_true": np.nan,
        "wgs_lat_true": np.nan,
        "coord_source": "",
        "coord_rule": "distance_gt_500m_reverse_not_matched_need_fallback",
        "needs_fallback": 1,
    })


chosen = reverse_detail.apply(choose_reverse_geocode_coord, axis=1)
for c in chosen.columns:
    reverse_detail[c] = chosen[c]

# 稳定 dtype，避免 pandas 3 批量回写时报 int8/string upcast 错误。
for c in ["gd_address_text", "bd_address_text", "coord_source", "coord_rule"]:
    if c not in df_merged.columns:
        df_merged[c] = ""
    df_merged[c] = df_merged[c].astype("object")
    reverse_detail[c] = reverse_detail[c].astype("object")

for c in ["gd_match_score", "bd_match_score", "wgs_lon_true", "wgs_lat_true"]:
    if c not in df_merged.columns:
        df_merged[c] = np.nan
    df_merged[c] = pd.to_numeric(df_merged[c], errors="coerce").astype("float64")
    reverse_detail[c] = pd.to_numeric(reverse_detail[c], errors="coerce").astype("float64")

reverse_detail["needs_fallback"] = (
    pd.to_numeric(reverse_detail["needs_fallback"], errors="coerce")
    .fillna(0)
    .astype("int64")
)

# 用 merge + 按列回填，避免 set_index 后 loc 批量赋值触发 dtype 错误。
reverse_cols = [
    "gd_address_text", "bd_address_text", "gd_match_score", "bd_match_score",
    "wgs_lon_true", "wgs_lat_true", "coord_source", "coord_rule", "needs_fallback",
]

reverse_update = (
    reverse_detail[["place_merge_id"] + reverse_cols]
    .drop_duplicates(subset=["place_merge_id"], keep="first")
    .copy()
)

df_merged = df_merged.merge(
    reverse_update,
    on="place_merge_id",
    how="left",
    suffixes=("", "_rev"),
)

for c in reverse_cols:
    rev = f"{c}_rev"
    if rev in df_merged.columns:
        mask = df_merged[rev].notna()

        if c in ["coord_source", "coord_rule", "gd_address_text", "bd_address_text"]:
            mask = mask & df_merged[rev].astype(str).str.len().gt(0)

        df_merged.loc[mask, c] = df_merged.loc[mask, rev]
        df_merged.drop(columns=[rev], inplace=True)

df_merged["needs_fallback"] = (
    pd.to_numeric(df_merged["needs_fallback"], errors="coerce")
    .fillna(0)
    .astype("int64")
)

unmatched_reverse = df_merged[df_merged["needs_fallback"].eq(1)].copy()

unmatched_reverse.to_csv(UNMATCHED_REVERSE_FILE, index=False, encoding="utf-8-sig")
reverse_detail.to_csv(REVERSE_MATCH_FILE, index=False, encoding="utf-8-sig")
df_merged.to_csv(PRE_FALLBACK_FILE, index=False, encoding="utf-8-sig")

print("Reverse-geocode fuzzy matching completed without additional API lookup.")
print(f"Reverse-match detail saved: {REVERSE_MATCH_FILE}")
print(f"Pre-fallback coordinate result saved: {PRE_FALLBACK_FILE}")
print(f"Unmatched reverse-geocode records saved: {UNMATCHED_REVERSE_FILE}")
print(f"Reverse-geocode matched records: {(reverse_detail['needs_fallback'] == 0).sum()}")
print(f"Need fallback records: {len(unmatched_reverse)}")

preview_cols = [c for c in ["companyid", "ENTNAME", "地区", "distance", "coord_rule"] if c in unmatched_reverse.columns]
print(unmatched_reverse[preview_cols].head())

  0%|          | 0/334 [1:09:24<?, ?it/s]


Reverse-geocode fuzzy matching completed without additional API lookup.
Reverse-match detail saved: D:\Hospital_Pharmacy program\data\hospital\hosp_registered_data\ol_data\05_added_hospital_reverse_geocode_match_detail.csv
Pre-fallback coordinate result saved: D:\Hospital_Pharmacy program\data\hospital\hosp_registered_data\ol_data\05_added_hospital_after_reverse_match.csv
Unmatched reverse-geocode records saved: D:\Hospital_Pharmacy program\data\hospital\hosp_registered_data\ol_data\05_added_hospital_reverse_unmatched_for_business_fill.csv
Reverse-geocode matched records: 31
Need fallback records: 1103
   companyid                  ENTNAME   地区      distance  \
0    1176224              福清市医院影像诊断中心  福清市  2.330914e+06   
1    2834800       泰安尚璟口腔医院有限公司尚璟口腔诊所  泰安市  1.497901e+06   
4   13401408  北京国医堂中医院集团有限公司国医堂祥云中医诊所  北京市  2.709882e+05   
5   18387520        成都嘉晨医院管理有限公司牧电路诊所  成都市  1.889595e+04   
6   19138960             深圳中仁_泌尿 外科医院  深圳市  1.314766e+06   

                             

In [13]:
# Output check after reverse-geocode fuzzy matching.
from pathlib import Path
import pandas as pd

BASE_DIR = Path(r"D:\Hospital_Pharmacy program\data\hospital\hosp_registered_data\ol_data")
PRE_FALLBACK_FILE = BASE_DIR / "05_added_hospital_after_reverse_match.csv"
REVERSE_MATCH_FILE = BASE_DIR / "05_added_hospital_reverse_geocode_match_detail.csv"
UNMATCHED_REVERSE_FILE = BASE_DIR / "05_added_hospital_reverse_unmatched_for_business_fill.csv"

for f in [PRE_FALLBACK_FILE, REVERSE_MATCH_FILE, UNMATCHED_REVERSE_FILE]:
    print(f, "exists=", f.exists(), "size=", f.stat().st_size if f.exists() else None)

if UNMATCHED_REVERSE_FILE.exists():
    unmatched = pd.read_csv(UNMATCHED_REVERSE_FILE, encoding="utf-8-sig", low_memory=False)
    print("Unmatched reverse-geocode rows needing fallback:", len(unmatched))
    display(unmatched.head())


D:\Hospital_Pharmacy program\data\hospital\hosp_registered_data\ol_data\05_added_hospital_after_reverse_match.csv exists= True size= 1583392
D:\Hospital_Pharmacy program\data\hospital\hosp_registered_data\ol_data\05_added_hospital_reverse_geocode_match_detail.csv exists= True size= 1446366
D:\Hospital_Pharmacy program\data\hospital\hosp_registered_data\ol_data\05_added_hospital_reverse_unmatched_for_business_fill.csv exists= True size= 942335
Unmatched reverse-geocode rows needing fallback: 1103


,companyid,PRIPID,ENTNAME,UNISCID,REGNO,NACAOID,NAME,NAMETITLE,ENTTYPE,ESDATE,...,place_merge_id,wgs_lon_true,wgs_lat_true,coord_source,coord_rule,needs_fallback,gd_address_text,bd_address_text,gd_match_score,bd_match_score
0,1176224,NaN,福清市医院影像诊断中心,NaN,3501811001684,NaN,陈性美,NaN,3200,1994/4/4,...,0,NaN,NaN,NaN,distance_gt_500m_reverse_not_matched_need_fall...,1,哈尔滨医科大学附属第六医院（伍联德医院） 爱婴大街 船口街道 哈尔滨市 黑龙江省,龙江街道 福州市 福清市 福建省,18.181818,35.294118
1,2834800,PROVINCENODENUM370000370902000022018122417963,泰安尚璟口腔医院有限公司尚璟口腔诊所,91370902MA3NWRYC65,NaN,MA3NWRYC6,王瑞元,负责人,2152,2018/12/24,...,1,NaN,NaN,NaN,distance_gt_500m_reverse_not_matched_need_fall...,1,水荫南路 黄花岗街道 越秀区 广东省,普照寺路 泰前街道 泰山区 山东省,0.000000,13.333333
2,13401408,PROVINCENODENUM110000ff8080815b27acb4015b3cb11...,北京国医堂中医院集团有限公司国医堂祥云中医诊所,91110113MA00D88P5L,110113023225233,MA00D88P5,张运杰,负责人,21501,2017/3/30,...,4,NaN,NaN,NaN,distance_gt_500m_reverse_not_matched_need_fall...,1,朝阳路 南街街道 朝阳市 辽宁省,京榆旧线辅路 潞邑街道 通州区,0.000000,13.333333
3,18387520,NaN,成都嘉晨医院管理有限公司牧电路诊所,91510105MA62Q35CX6,510105000664535,MA62Q35CX,庞玉芬,NaN,2130,2017/2/6,...,5,NaN,NaN,NaN,distance_gt_500m_reverse_not_matched_need_fall...,1,牧电路 草堂街道 青羊区 四川省,蜀源大道 团结街道 郫都区 四川省,37.500000,11.111111
4,19138960,NaN,深圳中仁_泌尿 外科医院,NaN,440310602422281,NaN,林国金,NaN,4540,2014/10/20,...,6,NaN,NaN,NaN,distance_gt_500m_reverse_not_matched_need_fall...,1,瓷都大道 黄土镇 夹江县 四川省,福中三路 莲花街道 福田区 广东省,0.000000,16.666667


In [15]:
import numpy as np
import pandas as pd
import eviltransform
from pathlib import Path

BASE_DIR = Path(r"D:\Hospital_Pharmacy program\data\hospital\hosp_registered_data\ol_data")
PRE_FALLBACK_FILE = BASE_DIR / "05_added_hospital_after_reverse_match.csv"
FINAL_FILE = BASE_DIR / "06_added_hospital_final_wgs_with_score.csv"
MISSING_FILE = BASE_DIR / "06_added_hospital_missing_coords.csv"
FALLBACK_AUDIT_FILE = BASE_DIR / "06_added_hospital_fallback_audit.csv"


def read_csv_auto(path):
    for enc in ["utf-8-sig", "utf-8", "gb18030", "gbk"]:
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(path, encoding="utf-8", errors="ignore", low_memory=False)


def pick_existing(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def valid_lon_lat(lon, lat):
    return (
        pd.notna(lon)
        and pd.notna(lat)
        and 70 <= float(lon) <= 140
        and 15 <= float(lat) <= 55
    )


def gcj_to_wgs(lng, lat):
    if not valid_lon_lat(lng, lat):
        return np.nan, np.nan
    wgs_lat, wgs_lng = eviltransform.gcj2wgs(float(lat), float(lng))
    return wgs_lng, wgs_lat


if not PRE_FALLBACK_FILE.exists():
    raise FileNotFoundError(f"Run the reverse-match cell first; missing {PRE_FALLBACK_FILE}")

df = read_csv_auto(PRE_FALLBACK_FILE)

BD_WGS_LON_COL = pick_existing(df, ["百度_wgs84经度", "百度_wgs84经度1"])
BD_WGS_LAT_COL = pick_existing(df, ["百度_wgs84纬度", "百度_wgs84纬度1"])
GD_WGS_LON_COL = pick_existing(df, ["高德_wgs84经度", "高德_wgs84经度1"])
GD_WGS_LAT_COL = pick_existing(df, ["高德_wgs84纬度", "高德_wgs84纬度1"])

REG_LON_COL = pick_existing(df, ["工商经度", "lng"])
REG_LAT_COL = pick_existing(df, ["工商纬度", "lat"])

REG_WGS_LON_COL = "工商_wgs84经度"
REG_WGS_LAT_COL = "工商_wgs84纬度"

for c in ["wgs_lon_true", "wgs_lat_true"]:
    if c not in df.columns:
        df[c] = np.nan
    df[c] = pd.to_numeric(df[c], errors="coerce")

for c in [BD_WGS_LON_COL, BD_WGS_LAT_COL, GD_WGS_LON_COL, GD_WGS_LAT_COL]:
    if c is not None and c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

if "score" not in df.columns:
    df["score"] = np.nan
if "coord_source" not in df.columns:
    df["coord_source"] = ""
if "coord_rule" not in df.columns:
    df["coord_rule"] = ""

df["coord_source"] = df["coord_source"].astype("object")
df["coord_rule"] = df["coord_rule"].astype("object")

# 已有高质量坐标评分
df.loc[df["coord_source"].eq("baidu"), "score"] = 10
df.loc[df["coord_source"].str.contains("reverse_geocode_match", na=False), "score"] = 9

# 1. 工商坐标按高德/GCJ-02 坐标处理，先转 WGS，再补仍缺失的坐标
if REG_LON_COL is not None and REG_LAT_COL is not None:
    df[REG_LON_COL] = pd.to_numeric(df[REG_LON_COL], errors="coerce")
    df[REG_LAT_COL] = pd.to_numeric(df[REG_LAT_COL], errors="coerce")

    df[[REG_WGS_LON_COL, REG_WGS_LAT_COL]] = df.apply(
        lambda row: gcj_to_wgs(row[REG_LON_COL], row[REG_LAT_COL]),
        axis=1,
        result_type="expand",
    )

    fill_business = (
        (df["wgs_lon_true"].isna() | df["wgs_lat_true"].isna())
        & df[REG_WGS_LON_COL].notna()
        & df[REG_WGS_LAT_COL].notna()
    )

    df.loc[fill_business, "wgs_lon_true"] = df.loc[fill_business, REG_WGS_LON_COL]
    df.loc[fill_business, "wgs_lat_true"] = df.loc[fill_business, REG_WGS_LAT_COL]
    df.loc[fill_business, "coord_source"] = "business_registration_gcj_to_wgs"
    df.loc[fill_business, "coord_rule"] = "reverse_unmatched_business_coordinate_fill"
    df.loc[fill_business, "score"] = 8
else:
    fill_business = pd.Series(False, index=df.index)
    print("No business-registration coordinate columns found; skipped business-coordinate filling.")

# 2. 最后兜底：先百度 WGS，再高德 WGS
if BD_WGS_LON_COL is not None and BD_WGS_LAT_COL is not None:
    fill_baidu_fallback = (
        (df["wgs_lon_true"].isna() | df["wgs_lat_true"].isna())
        & df[BD_WGS_LON_COL].notna()
        & df[BD_WGS_LAT_COL].notna()
    )

    df.loc[fill_baidu_fallback, "wgs_lon_true"] = df.loc[fill_baidu_fallback, BD_WGS_LON_COL]
    df.loc[fill_baidu_fallback, "wgs_lat_true"] = df.loc[fill_baidu_fallback, BD_WGS_LAT_COL]
    df.loc[fill_baidu_fallback, "coord_source"] = "baidu_wgs_fallback"
    df.loc[fill_baidu_fallback, "coord_rule"] = "final_fallback_baidu_wgs"
    df.loc[fill_baidu_fallback, "score"] = 6
else:
    fill_baidu_fallback = pd.Series(False, index=df.index)
    print("No Baidu WGS coordinate columns found; skipped Baidu fallback.")

if GD_WGS_LON_COL is not None and GD_WGS_LAT_COL is not None:
    fill_gaode_fallback = (
        (df["wgs_lon_true"].isna() | df["wgs_lat_true"].isna())
        & df[GD_WGS_LON_COL].notna()
        & df[GD_WGS_LAT_COL].notna()
    )

    df.loc[fill_gaode_fallback, "wgs_lon_true"] = df.loc[fill_gaode_fallback, GD_WGS_LON_COL]
    df.loc[fill_gaode_fallback, "wgs_lat_true"] = df.loc[fill_gaode_fallback, GD_WGS_LAT_COL]
    df.loc[fill_gaode_fallback, "coord_source"] = "gaode_wgs_fallback"
    df.loc[fill_gaode_fallback, "coord_rule"] = "final_fallback_gaode_wgs"
    df.loc[fill_gaode_fallback, "score"] = 5
else:
    fill_gaode_fallback = pd.Series(False, index=df.index)
    print("No Gaode WGS coordinate columns found; skipped Gaode fallback.")

missing = df[df["wgs_lon_true"].isna() | df["wgs_lat_true"].isna()].copy()

audit_cols = [
    "place_merge_id", "companyid", "ENTNAME", "distance", "coord_source", "coord_rule", "score",
    "wgs_lon_true", "wgs_lat_true",
    REG_WGS_LON_COL, REG_WGS_LAT_COL,
    BD_WGS_LON_COL, BD_WGS_LAT_COL,
    GD_WGS_LON_COL, GD_WGS_LAT_COL,
]
audit_cols = [c for c in audit_cols if c is not None and c in df.columns]

df.to_csv(FINAL_FILE, index=False, encoding="utf-8-sig")
missing.to_csv(MISSING_FILE, index=False, encoding="utf-8-sig")
df[audit_cols].to_csv(FALLBACK_AUDIT_FILE, index=False, encoding="utf-8-sig")

valid_wgs_count = int((df["wgs_lon_true"].notna() & df["wgs_lat_true"].notna()).sum())

print("Final coordinate selection completed without additional API lookup.")
print("Rows filled by business coordinates:", int(fill_business.sum()))
print("Rows filled by Baidu WGS fallback:", int(fill_baidu_fallback.sum()))
print("Rows filled by Gaode WGS fallback:", int(fill_gaode_fallback.sum()))
print("Final valid WGS rows:", valid_wgs_count)
print("Still missing coordinate rows:", len(missing))
print("Final file saved:", FINAL_FILE)
print("Missing file saved:", MISSING_FILE)
print("Audit file saved:", FALLBACK_AUDIT_FILE)

Final coordinate selection completed without additional API lookup.
Rows filled by business coordinates: 631
Rows filled by Baidu WGS fallback: 638
Rows filled by Gaode WGS fallback: 2
Final valid WGS rows: 1994
Still missing coordinate rows: 0
Final file saved: D:\Hospital_Pharmacy program\data\hospital\hosp_registered_data\ol_data\06_added_hospital_final_wgs_with_score.csv
Missing file saved: D:\Hospital_Pharmacy program\data\hospital\hosp_registered_data\ol_data\06_added_hospital_missing_coords.csv
Audit file saved: D:\Hospital_Pharmacy program\data\hospital\hosp_registered_data\ol_data\06_added_hospital_fallback_audit.csv


In [17]:
# Final output check.
from pathlib import Path
import pandas as pd

BASE_DIR = Path(r"D:\Hospital_Pharmacy program\data\hospital\hosp_registered_data\ol_data")
FINAL_FILE = BASE_DIR / "06_added_hospital_final_wgs_with_score.csv"
MISSING_FILE = BASE_DIR / "06_added_hospital_missing_coords.csv"
FALLBACK_AUDIT_FILE = BASE_DIR / "06_added_hospital_fallback_audit.csv"

for f in [FINAL_FILE, MISSING_FILE, FALLBACK_AUDIT_FILE]:
    print(f, "exists=", f.exists(), "size=", f.stat().st_size if f.exists() else None)

if FINAL_FILE.exists():
    final_df = pd.read_csv(FINAL_FILE, encoding="utf-8-sig", low_memory=False)

    valid_wgs_count = int(
        (final_df["wgs_lon_true"].notna() & final_df["wgs_lat_true"].notna()).sum()
    )

    print("Final rows:", len(final_df))
    print("Valid WGS rows:", valid_wgs_count)

    if "coord_source" in final_df.columns:
        display(final_df["coord_source"].value_counts(dropna=False))

    display(final_df.head())

D:\Hospital_Pharmacy program\data\hospital\hosp_registered_data\ol_data\06_added_hospital_final_wgs_with_score.csv exists= True size= 1691448
D:\Hospital_Pharmacy program\data\hospital\hosp_registered_data\ol_data\06_added_hospital_missing_coords.csv exists= True size= 610
D:\Hospital_Pharmacy program\data\hospital\hosp_registered_data\ol_data\06_added_hospital_fallback_audit.csv exists= True size= 510740
Final rows: 1994
Valid WGS rows: 1994


coord_source
baidu                               692
baidu_wgs_fallback                  638
business_registration_gcj_to_wgs    631
baidu_reverse_geocode_match          19
gaode_reverse_geocode_match          12
gaode_wgs_fallback                    2
Name: count, dtype: int64

,companyid,PRIPID,ENTNAME,UNISCID,REGNO,NACAOID,NAME,NAMETITLE,ENTTYPE,ESDATE,...,coord_source,coord_rule,needs_fallback,gd_address_text,bd_address_text,gd_match_score,bd_match_score,score,工商_wgs84经度,工商_wgs84纬度
0,1176224,NaN,福清市医院影像诊断中心,NaN,3501811001684,NaN,陈性美,NaN,3200,1994/4/4,...,business_registration_gcj_to_wgs,reverse_unmatched_business_coordinate_fill,1,哈尔滨医科大学附属第六医院（伍联德医院） 爱婴大街 船口街道 哈尔滨市 黑龙江省,龙江街道 福州市 福清市 福建省,18.181818,35.294118,8.0,119.378110,25.729885
1,2834800,PROVINCENODENUM370000370902000022018122417963,泰安尚璟口腔医院有限公司尚璟口腔诊所,91370902MA3NWRYC65,NaN,MA3NWRYC6,王瑞元,负责人,2152,2018/12/24,...,business_registration_gcj_to_wgs,reverse_unmatched_business_coordinate_fill,1,水荫南路 黄花岗街道 越秀区 广东省,普照寺路 泰前街道 泰山区 山东省,0.000000,13.333333,8.0,117.198636,36.232963
2,4653840,PROVINCENODENUM210000210103001012021061839646,沈阳脑康心理专科医院有限公司沈河精神卫生诊所,91210103MA116BC255,NaN,MA116BC25,苏珍发,负责人,2100,2021/6/28,...,business_registration_gcj_to_wgs,reverse_unmatched_business_coordinate_fill,0,NaN,NaN,NaN,NaN,8.0,123.422831,41.804883
3,10846160,NaN,易门矿务局职工医院木奔卫生所,NaN,5304251000024,NaN,巢洪仁,NaN,4410,1999/11/4,...,business_registration_gcj_to_wgs,reverse_unmatched_business_coordinate_fill,0,NaN,NaN,NaN,NaN,8.0,101.946455,24.628036
4,13401408,PROVINCENODENUM110000ff8080815b27acb4015b3cb11...,北京国医堂中医院集团有限公司国医堂祥云中医诊所,91110113MA00D88P5L,110113023225233,MA00D88P5,张运杰,负责人,21501,2017/3/30,...,business_registration_gcj_to_wgs,reverse_unmatched_business_coordinate_fill,1,朝阳路 南街街道 朝阳市 辽宁省,京榆旧线辅路 潞邑街道 通州区,0.000000,13.333333,8.0,116.537199,40.086746
